# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide to load, explore, and analyze the FAIR^2 mass spectrometry dataset, using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. All entities (record sets, fields, columns) are referenced by their `@id` as defined in the dataset's Croissant schema.

### Dataset Source
The dataset schema is accessible at: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed in your environment
!pip install -U mlcroissant pandas

## 1. Data Loading
Load dataset metadata and initialize the Croissant dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define URL to the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata - note: dataset.metadata behaves as an object, not a dict
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
The FAIR^2 Croissant schema may have multiple record sets. This section lists all record sets in the dataset with their `@id`, and for each record set, its available fields/columns (`@id`, name, and type, if available).

In [ ]:
# List all record sets with their @id and names
print("Available record sets in the dataset:")
for rs in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {rs.id} | name: {getattr(rs, 'name', '')}")
    # For each record set, show its fields and columns
    print("   Fields / Columns:")
    all_fields = getattr(rs, 'fields', [])
    all_columns = getattr(rs, 'columns', [])
    # 'fields' are logical, 'columns' are typically for tabular
    for fld in all_fields:
        print(f"     * field @id: {fld.id}, name: {getattr(fld, 'name', '')}, type: {getattr(fld, 'data_type', '')}")
    for col in all_columns:
        print(f"     * column @id: {col.id}, name: {getattr(col, 'name', '')}, type: {getattr(col, 'data_type', '')}")
    print()

## 3. Data Extraction
Load the records from each record set as a DataFrame for processing. All accesses use the `@id` of the record set as shown above. To demonstrate, we will pick the first record set and show its content.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f'All record set @ids: {record_set_ids}')

# Initialize a dictionary to hold the loaded DataFrames
dataframes = {}

# We'll extract data for all record sets (could be more than one)
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# Preview the first record set's columns and data
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nColumns in record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
We'll perform typical analysis steps: filtering, normalization, and grouping, all via fields referenced by their `@id`. Select a numeric field (`@id`), apply a filter, normalize, and (if suitable) group by a string/categorical field.

In [ ]:
import numpy as np

# For demonstration: Pick the first record set and list its columns
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Try to pick a numeric field; fallback to the first int/float column
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields detected: {numeric_candidates}")
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        raise ValueError("No numeric fields found in DataFrame.")

    # Filtering (e.g., protein abundance > threshold)
    threshold = df[numeric_field].quantile(0.75)  # top quartile as an example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()

    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a string/categorical field
    string_candidates = df.select_dtypes(include=[object]).columns.tolist()
    group_field = None
    for c in string_candidates:
        # Exclude columns with all unique values (likely IDs)
        if df[c].nunique() < (0.5 * len(df)):
            group_field = c
            break

    if group_field:
        print(f"\nGrouping by '{group_field}':")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and relationship with group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if record_set_ids:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()

# Boxplot by group_field
if record_set_ids and group_field:
    plt.figure(figsize=(10, 6))
    top_cats = df[group_field].value_counts().index[:10]
    sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_cats)])
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field} (Top 10)")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded the FAIR^2 mass spectrometry dataset using `mlcroissant`, referencing all entities by their `@id`.
- We explored available record sets, fields, and columns.
- A sample numeric field was filtered and normalized, and records were grouped where relevant.
- Visualizations revealed salient distributions and, where possible, groupwise trends.

You can adapt these steps for other record sets or fields by referencing the `@id`s from the overview section.